
## Phase 1C: Dataset Integration and Feature Engineering

### Objective

Merge the cleaned road complaints, weather, traffic,  datasets into a single machine learning dataset for model training.

In [32]:
# Import required libraries

import pandas as pd
import numpy as np


In [33]:
# Load datasets

road_complaints = pd.read_csv("../data/processed/road_complaints.csv")
weather = pd.read_csv("../data/processed/weather.csv")
traffic = pd.read_csv("../data/processed/traffic.csv")


In [34]:
# Convert date columns

road_complaints["Created Date"] = pd.to_datetime(road_complaints["Created Date"])
weather["time"] = pd.to_datetime(weather["time"])
traffic["Date"] = pd.to_datetime(traffic["Date"])

In [35]:
# Create date features for 311 complaints

road_complaints["Year"] = road_complaints["Created Date"].dt.year
road_complaints["Month"] = road_complaints["Created Date"].dt.month
road_complaints["Day"] = road_complaints["Created Date"].dt.day
road_complaints["Hour"] = road_complaints["Created Date"].dt.hour
road_complaints["DayOfWeek"] = road_complaints["Created Date"].dt.day_name()

In [36]:
# Standardize street names

road_complaints["Street Name"] = (
    road_complaints["Street Name"]
    .astype(str)
    .str.upper()
    .str.strip()
)

traffic["Roadway Name"] = (
    traffic["Roadway Name"]
    .astype(str)
    .str.upper()
    .str.strip()
)

In [37]:
# Create monthly weather features

weather["Year"] = weather["time"].dt.year
weather["Month"] = weather["time"].dt.month

monthly_weather = (
    weather
    .groupby(["Year", "Month"], as_index=False)
    .agg({
        "temperature_2m (°F)": "mean",
        "precipitation (inch)": "sum",
        "snowfall (inch)": "sum",
        "snow_depth (ft)": "mean",
        "weather_code (wmo code)": "first",
        "wind_speed_10m (mp/h)": "mean"
    })
)

In [38]:
# Create total traffic volume

hour_columns = [
    col for col in traffic.columns
    if ":" in col
]

traffic[hour_columns] = traffic[hour_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

traffic["Total Traffic"] = traffic[hour_columns].sum(axis=1)

In [39]:
# Create monthly traffic features

traffic["Year"] = traffic["Date"].dt.year
traffic["Month"] = traffic["Date"].dt.month

monthly_traffic = (
    traffic
    .groupby(["Roadway Name", "Year", "Month"], as_index=False)
    .agg({
        "Total Traffic": "mean"
    })
)

In [40]:
# Create monthly road complaint counts

monthly_complaints = (
    road_complaints
    .groupby(["Street Name", "Borough", "Year", "Month"], as_index=False)
    .agg({
        "Unique Key": "count",
        "Latitude": "mean",
        "Longitude": "mean"
    })
)

monthly_complaints = monthly_complaints.rename(
    columns={"Unique Key": "Complaint Count"}
)

In [41]:
# Merge complaint data with weather data

model_data = monthly_complaints.merge(
    monthly_weather,
    on=["Year", "Month"],
    how="left"
)

model_data.shape

(311796, 13)

In [42]:
# Merge complaint data with traffic data

model_data = model_data.merge(
    monthly_traffic,
    left_on=["Street Name", "Year", "Month"],
    right_on=["Roadway Name", "Year", "Month"],
    how="left"
)

model_data.shape

(311796, 15)

In [43]:
# Fill missing traffic values

model_data["Total Traffic"] = model_data["Total Traffic"].fillna(0)

In [44]:
# Create maintenance risk target

risk_threshold = model_data["Complaint Count"].quantile(0.75)

model_data["Maintenance Risk"] = np.where(
    model_data["Complaint Count"] >= risk_threshold,
    1,
    0
)

In [45]:
# Check target distribution

model_data["Maintenance Risk"].value_counts()

Maintenance Risk
0    222519
1     89277
Name: count, dtype: int64

In [46]:
# Check missing values

model_data.isnull().sum()

Street Name                     0
Borough                         0
Year                            0
Month                           0
Complaint Count                 0
Latitude                    26601
Longitude                   26601
temperature_2m (°F)             0
precipitation (inch)            0
snowfall (inch)                 0
snow_depth (ft)                 0
weather_code (wmo code)         0
wind_speed_10m (mp/h)           0
Roadway Name               309797
Total Traffic                   0
Maintenance Risk                0
dtype: int64

In [47]:
# Preview final dataset

model_data.head()

,Street Name,Borough,Year,Month,Complaint Count,Latitude,Longitude,temperature_2m (°F),precipitation (inch),snowfall (inch),snow_depth (ft),weather_code (wmo code),wind_speed_10m (mp/h),Roadway Name,Total Traffic,Maintenance Risk
0,0CEAN AVENUE,BROOKLYN,2013,8,1,NaN,NaN,72.017876,2.349,0.000,0.000000,3,5.590860,NaN,0.000000,0
1,1 AVE,MANHATTAN,2012,10,1,NaN,NaN,57.447177,4.651,0.000,0.000000,0,7.310349,1 AVE,7402.357143,0
2,1 AVE,MANHATTAN,2014,7,1,NaN,NaN,74.537366,3.384,0.000,0.000000,3,6.315726,NaN,0.000000,0
3,1 AVE,MANHATTAN,2019,3,1,NaN,NaN,38.369624,4.445,11.028,0.128503,3,9.210618,NaN,0.000000,0
4,1 AVE,UNSPECIFIED,2018,6,1,NaN,NaN,69.343611,3.557,0.000,0.000000,3,7.394444,NaN,0.000000,0


In [48]:
# Check final dataset information

model_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 311796 entries, 0 to 311795
Data columns (total 16 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Street Name              311796 non-null  str    
 1   Borough                  311796 non-null  str    
 2   Year                     311796 non-null  int32  
 3   Month                    311796 non-null  int32  
 4   Complaint Count          311796 non-null  int64  
 5   Latitude                 285195 non-null  float64
 6   Longitude                285195 non-null  float64
 7   temperature_2m (°F)      311796 non-null  float64
 8   precipitation (inch)     311796 non-null  float64
 9   snowfall (inch)          311796 non-null  float64
 10  snow_depth (ft)          311796 non-null  float64
 11  weather_code (wmo code)  311796 non-null  int64  
 12  wind_speed_10m (mp/h)    311796 non-null  float64
 13  Roadway Name             1999 non-null    str    
 14  Total Traffic  

In [49]:
# Remove Roadway Name since it is only used during merging
model_data = model_data.drop(columns=["Roadway Name"])

In [52]:
# Save final machine learning dataset

model_data.to_csv(
    "../data/processed/model_data.csv",
    index=False
)

In [53]:
# Verify saved dataset

final_data = pd.read_csv("../data/processed/model_data.csv")

final_data.shape

(311796, 15)

## Phase 1C Summary (Continued)

New machine learning features were created by combining the cleaned complaint, weather, and traffic datasets. Time-based features, weather summaries, traffic information, and the target variable, **Maintenance Risk**, were generated to produce the final dataset used for model training.